In [3]:
import torch; 
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)

## LeNet-5 Adaptado

- Ajustar el tamaño de entrada a 64×64×3
- Mantener la estructura original: 2 bloques conv-pool + 3 capas FC
- Añadir una variante con Batch Normalization después de cada capa convolucional

### Arquitectura
| Capa | Tipo | Filtros | Kernel | Salida |
|------|------|---------|--------|--------|
| Input | - | - | - | (3, 64, 64) |
| C1 | Conv2D | 6 | 5x5 | (6, 60, 60) |
| S2 | MaxPool | - | 2x2 | (6, 30, 30) |
| C3 | Conv2D | 16 | 5x5 | (16, 26, 26) |
| S4 | MaxPool | - | 2x2 | (16, 13, 13) |
| Flatten | - | - | - | 2704 |
| FC1 | Linear | - | - | 120 |
| FC2 | Linear | - | - | 84 |
| Output | Linear | - | - | 4 |

In [5]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        # Capas
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(in_features=2704, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.out = nn.Linear(in_features=84, out_features=4)
        
    def forward(self, x):
        # Flujo
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

In [6]:
model = LeNet5()
x = torch.randn(1, 3, 64, 64)
print(model(x).shape)

torch.Size([1, 4])


In [7]:
class LeNet5BN(nn.Module):
    def __init__(self):
        super().__init__()
        # Capas
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.bn1 = nn.BatchNorm2d(6)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.bn2 = nn.BatchNorm2d(16)
        self.fc1 = nn.Linear(in_features=2704, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.out = nn.Linear(in_features=84, out_features=4)
        
    def forward(self, x):
        # Flujo
        x = F.relu(self.conv1(x))
        x = self.bn1(x)
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.bn2(x)
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

In [8]:
model_bn = LeNet5BN()
x = torch.randn(1, 3, 64, 64)
print(model_bn(x).shape)

torch.Size([1, 4])


## VGG-11 Simplificado
### Arquitectura 
| Capa | Tipo | Filtros | Kernel | Salida |
|------|------|---------|--------|--------|
| Input | - | - | - | (3, 64, 64) |
| Conv1 | Conv2D | 32 | 3x3 | (32, 64, 64) |
| Pool1 | MaxPool | - | 2x2 | (32, 32, 32) |
| Conv2 | Conv2D | 64 | 3x3 | (64, 32, 32) |
| Pool2 | MaxPool | - | 2x2 | (64, 16, 16) |
| Conv3 | Conv2D | 128 | 3x3 | (128, 16, 16) |
| Conv4 | Conv2D | 128 | 3x3 | (128, 16, 16) |
| Pool3 | MaxPool | - | 2x2 | (128, 8, 8) |
| Conv5 | Conv2D | 256 | 3x3 | (256, 8, 8) |
| Conv6 | Conv2D | 256 | 3x3 | (256, 8, 8) |
| Pool4 | MaxPool | - | 2x2 | (256, 4, 4) |
| Conv7 | Conv2D | 256 | 3x3 | (256, 4, 4) |
| Conv8 | Conv2D | 256 | 3x3 | (256, 4, 4) |
| Pool5 | MaxPool | - | 2x2 | (256, 2, 2) |
| Flatten | - | - | - | 1024 |
| FC1 | Linear | - | - | 2048 |
| FC2 | Linear | - | - | 2048 |
| Output | Linear | - | - | 4 |

In [ ]:
class VGG11(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(in_features=1024, out_features=2048)
        self.fc2 = nn.Linear(in_features=2048, out_features=2048)
        self.out = nn.Linear(in_features=2048, out_features=4)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = self.pool(x)
        x = F.relu(self.conv7(x))
        x = F.relu(self.conv8(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        
        return x



In [ ]:
model_bn = LeNet5BN()
x = torch.randn(1, 3, 64, 64)
print(model_bn(x).shape)

torch.Size([1, 4])
